In [ ]:
import numpy as np
import pandas as pd
import re
import yaml
from pathlib import Path
from tqdm import tqdm

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## WOS journal

In [ ]:
wos_path = Path(dataset_config['path_wos'] + 'WOS_OpenAlex/all_articles/')

In [ ]:
# Import OpenAlex data. Executive time: 2 min
all_wos = []
for p in wos_path.iterdir():
    print('Loading', p)
    df_item = pd.read_csv(p, usecols=['issn', 'journal'])
    all_wos.append(df_item)

In [ ]:
WOS_journal = pd.concat(all_wos).drop_duplicates(subset=['issn']).dropna(subset=['issn'])
WOS_journal.rename(columns={"issn": "issn_WOS", "journal": "journal_WOS"}, inplace=True)
WOS_journal # WOS: 35,777 ISSN

## CNKI journal

In [ ]:
CNKI_journal = pd.read_csv(dataset_config['path_cnki'] +'paper_list.txt', encoding='utf-8', delimiter='|', usecols=['issn', 'journal']).drop_duplicates(subset=['issn']).dropna(subset=['issn'])
CNKI_journal.rename(columns={"issn": "issn_CNKI", "journal": "journal_CNKI"}, inplace=True)
CNKI_journal # CNKI: 3062 ISSN

## ISSN overlap

In [ ]:
# Find overlap between WOS_journal and CNKI_journal by ISSN
overlap_df = pd.merge(
    WOS_journal,
    CNKI_journal,
    left_on='issn_WOS',
    right_on='issn_CNKI',
    how='inner'
)

# Calculate overlap statistics
num_overlap = len(overlap_df)
num_wos = len(WOS_journal)
num_cnki = len(CNKI_journal)

percent_wos = num_overlap / num_wos * 100
percent_cnki = num_overlap / num_cnki * 100

# Print results
print(f"Number of overlapping ISSNs: {num_overlap}")
print(f"Overlap percentage (of WOS): {percent_wos:.2f}%")
print(f"Overlap percentage (of CNKI): {percent_cnki:.2f}%")

print(overlap_df.head())

overlap_df.to_csv(dataset_config['path_processed'] + "CNKI/overlap_cnkiwos_issn.csv", index=False)

In [ ]:
# Function to check if a string contains Chinese characters
def has_chinese(text):
    return bool(re.search(r'[\u4e00-\u9fff]', str(text)))

# Apply to the CNKI journal names
overlap_df['is_chinese'] = overlap_df['journal_CNKI'].apply(has_chinese)

# Calculate counts
num_total = len(overlap_df)
num_chinese = overlap_df['is_chinese'].sum()
num_english = num_total - num_chinese

# Calculate percentages
pct_chinese = num_chinese / num_total * 100
pct_english = num_english / num_total * 100

# Display results
print(f"Total overlap journals: {num_total}")
print(f"Chinese journal names in CNKI: {num_chinese} ({pct_chinese:.2f}%)")
print(f"English journal names in CNKI: {num_english} ({pct_english:.2f}%)")

# Optional: preview samples
print("\nExamples of Chinese CNKI journal names:")
print(overlap_df[overlap_df['is_chinese']].head(5))
print("\nExamples of English CNKI journal names:")
print(overlap_df[~overlap_df['is_chinese']].head(5))